In [1]:
!pip install -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.17.0
    Uninstalling transformers-5.17.0:
      Successfully uninstalled transformers-5.17.0


In [2]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)

GPU: Tesla T4
VRAM: 14.56 GB


In [3]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Important for batching
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Left padding is preferable for batched causal-LM generation
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()

print("Model loaded successfully!")
print("Device:", model.device)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model loaded successfully!
Device: cuda:0


In [4]:
print(torch.cuda.get_device_name(0))

Tesla T4


In [5]:
messages = [
    {
        "role": "user",
        "content": "Respond with exactly one word: READY"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

input_ids = inputs["input_ids"].to(model.device)
attention_mask = inputs["attention_mask"].to(model.device)

with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=10,
        do_sample=False
    )

generated_tokens = outputs[0][input_ids.shape[-1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("Model response:")
print(response)

Model response:
READY


In [ ]:
df = pd.read_csv("/content/data_6k.csv")

# Already corrected 1,000 rows
checkpoint_df = pd.read_csv(
    "/content/qwen_label_corrections_checkpoint.csv"
)

print("Original dataset:", len(df))
print("Already corrected:", len(checkpoint_df))

Original dataset: 6000
Already corrected: 1000


In [ ]:
processed_ids = set(
    checkpoint_df["row_id"].astype(int)
)

remaining_df = df[
    ~df.index.isin(processed_ids)
].copy()

remaining_df = remaining_df.reset_index(drop=False)

remaining_df.rename(
    columns={"index": "row_id"},
    inplace=True
)

print("Remaining rows:", len(remaining_df))

Remaining rows: 5000


In [ ]:
classes = df["Toxic Category"].dropna().unique().tolist()

print(classes)

['Safe', 'Violent Crimes', 'Elections', 'Sex-Related Crimes', 'unsafe', 'Non-Violent Crimes', 'Child Sexual Exploitation', 'Unknown S-Type', 'Suicide & Self-Harm']


In [6]:
CLASS_DEFINITIONS = {
    "Safe": "Benign content with no harmful or illegal activity.",

    "Violent Crimes": "Violent criminal activity against people, such as murder, assault, kidnapping, or terrorism.",

    "Non-Violent Crimes": "Non-violent criminal or illegal activity, such as theft, fraud, burglary, hacking, or scams.",

    "unsafe": "Harmful, dangerous, or inappropriate content that does not clearly fit a more specific category.",

    "Unknown S-Type": "Content whose category cannot be reliably determined from the text.",

    "Sex-Related Crimes": "Sexual crimes or illegal sexual activity.",

    "Suicide & Self-Harm": "Suicide, suicidal ideation, self-harm, or content specifically involving harming oneself.",

    "Elections": "Content involving elections, voting, candidates, campaigns, or election-related activity.",

    "Child Sexual Exploitation": "Sexual exploitation, abuse, grooming, or sexual material involving children or minors."
}

In [7]:
LABELS_TEXT = "\n".join(
    f"- {label}: {definition}"
    for label, definition in CLASS_DEFINITIONS.items()
)

In [ ]:
print(set(df["Toxic Category"].unique()) - set(CLASS_DEFINITIONS.keys()))
print(set(CLASS_DEFINITIONS.keys()) - set(df["Toxic Category"].unique()))

set()
set()


In [ ]:
def build_prompt(text, existing_labels):

    if isinstance(existing_labels, list):
        labels = ", ".join(existing_labels)
    else:
        labels = str(existing_labels)

    return f"""
You are correcting labels in a text safety classification dataset.

Choose exactly ONE category from the nine categories below.

{LABELS_TEXT}

Rules:
- Judge the meaning of the text, not isolated keywords.
- Do not invent categories.
- Prefer specific categories when clearly supported.
- Child Sexual Exploitation takes priority for sexual exploitation involving minors.
- Suicide & Self-Harm takes priority for suicide or self-harm.
- Violent Crimes is for primarily violent criminal activity.
- Non-Violent Crimes is for primarily non-violent criminal activity.
- Safe is for clearly benign content.
- Unknown S-Type is only for content that cannot reasonably be classified.
- The existing label may be wrong.

Text:
{text}

Existing label(s):
{labels}

Return ONLY valid JSON:

{{
  "corrected_label": "one category",
  "confidence": 0.95,
  "reason": "very brief reason"
}}
"""

In [ ]:
print("Remaining rows:", len(remaining_df))
print("Unique texts:", remaining_df["text"].nunique())
print("Duplicate rows:", remaining_df["text"].duplicated().sum())

Remaining rows: 5000
Unique texts: 2021
Duplicate rows: 2979


In [ ]:
unique_texts = (
    remaining_df
    .groupby("text", as_index=False)
    .agg(
        row_ids=("row_id", list),
        existing_labels=("Toxic Category", lambda x: list(set(x))),
        duplicate_count=("row_id", "count")
    )
)

print("Original rows:", len(remaining_df))
print("Unique texts:", len(unique_texts))

print(
    "Texts with conflicting labels:",
    (unique_texts["existing_labels"].str.len() > 1).sum()
)

Original rows: 5000
Unique texts: 2021
Texts with conflicting labels: 3


In [ ]:
def ask_qwen_batch(prompts):

    texts = [
        tokenizer.apply_chat_template(
            [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            tokenize=False,
            add_generation_prompt=True
        )
        for prompt in prompts
    ]

    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=45,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id
        )

    # Because the inputs are padded, use the padded sequence length
    input_length = inputs["input_ids"].shape[1]

    responses = []

    for i in range(len(prompts)):

        generated_tokens = outputs[i][input_length:]

        response = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        responses.append(response)

    return responses

In [ ]:

BATCH_SIZE = 16

qwen_results = []

for start in tqdm(
    range(0, len(unique_texts), BATCH_SIZE),
    desc="Qwen label correction"
):

    batch = unique_texts.iloc[
        start:start + BATCH_SIZE
    ]

    prompts = []

    for _, row in batch.iterrows():

        prompts.append(
            build_prompt(
                text=str(row["text"]),
                existing_labels=row["existing_labels"]
            )
        )

    responses = ask_qwen_batch(prompts)

    for (_, row), response in zip(
        batch.iterrows(),
        responses
    ):

        result = parse_response(response)

        if result is None:

            result = {
                "corrected_label": row["existing_labels"][0],
                "confidence": 0,
                "reason": "Could not parse Qwen response"
            }

        corrected_label = result.get(
            "corrected_label",
            row["existing_labels"][0]
        )

        # Prevent invalid labels
        if corrected_label not in CLASS_DEFINITIONS:

            corrected_label = row["existing_labels"][0]

            result["confidence"] = 0
            result["reason"] = "Invalid label returned by Qwen"

        qwen_results.append({

            "text": row["text"],

            "row_ids": row["row_ids"],

            "existing_labels": row["existing_labels"],

            "corrected_label": corrected_label,

            "confidence": result.get(
                "confidence",
                0
            ),

            "reason": result.get(
                "reason",
                ""
            ),

            "raw_response": response

        })

Qwen label correction:   0%|          | 0/127 [00:00<?, ?it/s]

In [ ]:
qwen_results_df = pd.DataFrame(qwen_results)

In [ ]:
label_map = dict(
    zip(
        qwen_results_df["text"],
        qwen_results_df["corrected_label"]
    )
)

In [ ]:
remaining_df["corrected_label"] = (
    remaining_df["text"]
    .map(label_map)
)

In [ ]:
remaining_df["label_changed"] = (
    remaining_df["Toxic Category"]
    != remaining_df["corrected_label"]
)

In [ ]:
print(
    "Labels changed:",
    remaining_df["label_changed"].sum()
)

print(
    "Labels unchanged:",
    (~remaining_df["label_changed"]).sum()
)

Labels changed: 1012
Labels unchanged: 3988


In [ ]:
remaining_df["Toxic Category"] = (
    remaining_df["corrected_label"]
)

In [ ]:
remaining_df = remaining_df.drop(
    columns=["corrected_label"]
)

In [ ]:
remaining_df.shape

(5000, 4)

In [ ]:
remaining_df.columns

Index(['row_id', 'text', 'Toxic Category', 'label_changed'], dtype='object')

In [ ]:
checkpoint_df.columns

Index(['row_id', 'text', 'original_label', 'corrected_label', 'label_changed',
       'confidence', 'reason', 'raw_response'],
      dtype='object')

In [ ]:
checkpoint_df.shape

(1000, 8)

In [ ]:
checkpoint_df.drop(["original_label", "confidence", "reason", "raw_response"], axis=1, inplace=True)

In [ ]:
checkpoint_df.columns

Index(['row_id', 'text', 'corrected_label', 'label_changed'], dtype='object')

In [ ]:
checkpoint_df["Toxic Category"] = (
    checkpoint_df["corrected_label"]
)

checkpoint_df = checkpoint_df.drop(
    columns=["corrected_label"]
)

In [ ]:
final_df = pd.concat(
    [checkpoint_df, remaining_df],
    ignore_index=True
)

print("Final rows:", len(final_df))
print(final_df["Toxic Category"].value_counts())

Final rows: 6000
Toxic Category
Safe                         1646
Violent Crimes               1610
Suicide & Self-Harm          1211
Non-Violent Crimes            553
unsafe                        455
Unknown S-Type                196
Sex-Related Crimes            115
Elections                     111
Child Sexual Exploitation     103
Name: count, dtype: int64


In [ ]:
final_df.duplicated().sum()

np.int64(0)

In [ ]:
final_df.head()

,row_id,text,label_changed,Toxic Category
0,0,A child playing in a sunny meadow.,False,Safe
1,1,A family enjoying a picnic in the park.,False,Safe
2,2,A child playing in a sunny meadow.,False,Safe
3,3,Police tape across a crime scene at night.,False,Violent Crimes
4,4,A child playing in a sunny meadow.,True,Safe


In [ ]:
final_df.drop(["label_changed", "row_id"], axis=1, inplace = True)

In [ ]:
final_df.duplicated().sum()

np.int64(3972)

In [ ]:
print("Rows:", len(final_df))

print("\nDuplicate texts:")
print(final_df["text"].duplicated().sum())

print("\nDuplicate text + label:")
print(
    final_df.duplicated(
        subset=["text", "Toxic Category"]
    ).sum()
)

Rows: 6000

Duplicate texts:
3979

Duplicate text + label:
3972


In [ ]:
final_df.to_csv(
    "/content/Final_before_generate.csv",
    index=False
)

In [8]:
import pandas as pd
import re

df = pd.read_csv("/content/Final_before_generate.csv")

def normalize_for_duplicate(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

df["normalized_text"] = df["text"].apply(
    normalize_for_duplicate
)

print("Rows:", len(df))

Rows: 6000


In [9]:
label_col = "Toxic Category"

conflicts = (
    df.groupby("normalized_text")[label_col]
      .nunique()
)

conflicting_texts = conflicts[
    conflicts > 1
].index

print(
    "Conflicting duplicate groups:",
    len(conflicting_texts)
)

Conflicting duplicate groups: 7


In [10]:
conflicting_df = df[
    df["normalized_text"].isin(conflicting_texts)
].sort_values("normalized_text")

display(
    conflicting_df[
        [
            "text",
            "Toxic Category"
        ]
    ]
)

,text,Toxic Category
0,A child playing in a sunny meadow.,Safe
2039,A child playing in a sunny meadow.,Suicide & Self-Harm
813,A child playing in a sunny meadow.,Safe
815,A child playing in a sunny meadow.,Safe
818,A child playing in a sunny meadow.,Safe
...,...,...
1262,Symbols representing danger or restricted areas.,unsafe
1110,Symbols representing danger or restricted areas.,unsafe
378,Symbols representing danger or restricted areas.,Safe
801,Symbols representing danger or restricted areas.,Safe


In [11]:
df_clean = df.drop_duplicates(
    subset=["normalized_text", label_col],
    keep="first"
).copy()

df_clean = df_clean.reset_index(drop=True)

print("Before:", len(df))
print("After:", len(df_clean))
print("Removed:", len(df) - len(df_clean))

Before: 6000
After: 2028
Removed: 3972


In [12]:
print(
    df_clean[label_col].value_counts()
)

Toxic Category
Safe                         884
Violent Crimes               695
Non-Violent Crimes           208
unsafe                       144
Unknown S-Type                84
Suicide & Self-Harm            6
Elections                      3
Sex-Related Crimes             2
Child Sexual Exploitation      2
Name: count, dtype: int64


In [13]:
df_clean = df_clean.drop(
    columns=["normalized_text"]
)

In [14]:
df_clean.head()

,text,Toxic Category
0,A child playing in a sunny meadow.,Safe
1,A family enjoying a picnic in the park.,Safe
2,Police tape across a crime scene at night.,Violent Crimes
3,An emergency vehicle with flashing lights at a...,Violent Crimes
4,People walking through a peaceful city street.,Safe


# Generate data

In [29]:
TARGET_PER_CLASS = 500

current_counts = combined_df[label_col].value_counts()

generation_plan = {}

for label in CLASS_DEFINITIONS.keys():

    current = current_counts.get(label, 0)

    needed = max(
        0,
        TARGET_PER_CLASS - current
    )

    generation_plan[label] = needed

print(generation_plan)

{'Safe': 0, 'Violent Crimes': 0, 'Non-Violent Crimes': 0, 'unsafe': np.int64(45), 'Unknown S-Type': np.int64(304), 'Sex-Related Crimes': np.int64(385), 'Suicide & Self-Harm': 0, 'Elections': np.int64(389), 'Child Sexual Exploitation': np.int64(397)}


In [32]:
import random

def get_real_examples(label, n=10):

    examples = df[
        (df["Toxic Category"] == label)
    ]["text"].dropna().astype(str).tolist()

    if len(examples) <= n:
        return examples

    return random.sample(examples, n)

In [18]:
def build_generation_prompt(
    target_label,
    examples,
    number_to_generate
):

    all_definitions = "\n\n".join(
        f"{label}: {definition.strip()}"
        for label, definition
        in CLASS_DEFINITIONS.items()
    )

    examples_text = "\n".join(
        f"- {example}"
        for example in examples
    )

    return f"""
You are generating synthetic data for a text classification dataset.

The dataset has these nine categories:

{all_definitions}

TARGET CATEGORY:
{target_label}

REAL EXAMPLES FROM THE TARGET CATEGORY:
{examples_text}

Generate {number_to_generate} NEW and DIFFERENT text samples
that belong specifically to:

{target_label}

IMPORTANT RULES:

1. Every generated sample MUST belong to the target category.
2. Do not copy the real examples.
3. Do not make simple word substitutions.
4. Make the examples meaningfully different from each other.
5. Do not generate examples from another category.
6. Do not mention that the text is synthetic.
7. Do not add explanations.
8. Do not add numbering.
9. Return ONLY a JSON array of strings.
10. Each string must be one independent dataset sample.

For categories involving sensitive content, keep examples
non-graphic and suitable for classification research.
For Child Sexual Exploitation, generate only non-explicit,
classification-oriented descriptions and do not generate sexual
content involving minors.

OUTPUT FORMAT:

[
    "sample 1",
    "sample 2",
    "sample 3"
]
"""

In [19]:
import json
import re
import torch

def generate_samples(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=1200,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = outputs[0][
        inputs["input_ids"].shape[-1]:
    ]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    return response

In [20]:
def parse_generated_samples(response):

    response = response.strip()

    response = re.sub(
        r"```json\s*",
        "",
        response,
        flags=re.IGNORECASE
    )

    response = re.sub(
        r"```\s*",
        "",
        response
    )

    match = re.search(
        r"\[.*\]",
        response,
        re.DOTALL
    )

    if not match:
        return []

    try:

        data = json.loads(
            match.group()
        )

        if isinstance(data, list):

            return [
                str(x).strip()
                for x in data
                if str(x).strip()
            ]

    except Exception:
        pass

    return []

In [33]:
GENERATION_BATCH = 20
MAX_RETRIES = 5

additional_synthetic = []

for label, needed in generation_plan.items():

    if needed == 0:
        continue

    print(f"\n===== {label} =====")

    generated_count = 0
    retries = 0

    while generated_count < needed:

        batch_size = min(
            GENERATION_BATCH,
            needed - generated_count
        )

        examples = get_real_examples(
            label,
            n=min(10, current_counts[label])
        )

        prompt = build_generation_prompt(
            target_label=label,
            examples=examples,
            number_to_generate=batch_size
        )

        response = generate_samples(prompt)

        samples = parse_generated_samples(response)

        if not samples:

            retries += 1

            print(
                f"Generation failed "
                f"({retries}/{MAX_RETRIES})"
            )

            if retries >= MAX_RETRIES:
                break

            continue

        retries = 0

        samples = samples[:batch_size]

        for text in samples:

            additional_synthetic.append({
                "text": text,
                "final_label": label,
                "source": "synthetic"
            })

        generated_count += len(samples)

        print(
            f"{generated_count}/{needed}"
        )


===== unsafe =====
20/45
40/45
45/45

===== Unknown S-Type =====
20/304
40/304
60/304
80/304
100/304
120/304
140/304
160/304
180/304
200/304
220/304
240/304
260/304
280/304
300/304
304/304

===== Sex-Related Crimes =====
20/385
40/385
60/385
80/385
100/385
120/385
140/385
160/385
180/385
200/385
220/385
240/385
260/385
280/385
300/385
320/385
340/385
360/385
380/385
385/385

===== Elections =====
20/389
40/389
60/389
80/389
100/389
120/389
140/389
160/389
180/389
200/389
220/389
240/389
260/389
280/389
300/389
320/389
340/389
360/389
380/389
389/389

===== Child Sexual Exploitation =====
20/397
40/397
60/397
80/397
100/397
120/397
140/397
160/397
180/397
200/397
220/397
240/397
260/397
280/397
300/397
320/397
340/397
360/397
380/397
397/397


In [23]:
synthetic_df = pd.DataFrame(
    synthetic_rows
)

print(
    "Generated:",
    len(synthetic_df)
)

Generated: 275


In [24]:
def normalize_text(text):

    text = str(text).lower().strip()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text

In [25]:
real_normalized = set(
    df["text"]
      .astype(str)
      .apply(normalize_text)
)

synthetic_df["normalized_text"] = (
    synthetic_df["text"]
    .apply(normalize_text)
)

# Remove duplicate synthetic samples
synthetic_df = (
    synthetic_df
    .drop_duplicates(
        subset=["normalized_text"]
    )
)

# Remove synthetic samples already present
# in the real dataset
synthetic_df = synthetic_df[
    ~synthetic_df["normalized_text"].isin(
        real_normalized
    )
].copy()

synthetic_df = synthetic_df.drop(
    columns=["normalized_text"]
)

synthetic_df.reset_index(
    drop=True,
    inplace=True
)

print(
    "Unique synthetic samples:",
    len(synthetic_df)
)

Unique synthetic samples: 273


In [26]:
combined_df = pd.concat(
    [
        df[["text", "Toxic Category"]].assign(
            source="real"
        ),
        synthetic_df
    ],
    ignore_index=True
)

print(
    combined_df["Toxic Category"].value_counts()
)

Toxic Category
Safe                         1646
Violent Crimes               1610
Suicide & Self-Harm          1211
Non-Violent Crimes            553
unsafe                        455
Unknown S-Type                196
Sex-Related Crimes            115
Elections                     111
Child Sexual Exploitation     103
Name: count, dtype: int64


In [27]:
combined_df.to_csv(
    "/content/toxic_data_with_synthetic.csv",
    index=False
)

synthetic_df.to_csv(
    "/content/synthetic_data_only.csv",
    index=False
)

In [34]:
additional_df = pd.DataFrame(
    additional_synthetic
)

additional_df["normalized_text"] = (
    additional_df["text"]
    .apply(normalize_text)
)

existing_texts = set(
    combined_df["text"]
    .astype(str)
    .apply(normalize_text)
)

additional_df = (
    additional_df
    .drop_duplicates(
        subset=["normalized_text"]
    )
)

additional_df = additional_df[
    ~additional_df["normalized_text"].isin(
        existing_texts
    )
].copy()

additional_df.drop(
    columns=["normalized_text"],
    inplace=True
)

print(
    "New unique synthetic:",
    len(additional_df)
)

New unique synthetic: 1485


In [35]:
combined_df = pd.concat(
    [
        combined_df,
        additional_df
    ],
    ignore_index=True
)

print(
    combined_df["Toxic Category"].value_counts()
)

Toxic Category
Safe                         1646
Violent Crimes               1610
Suicide & Self-Harm          1211
Non-Violent Crimes            553
unsafe                        455
Unknown S-Type                196
Sex-Related Crimes            115
Elections                     111
Child Sexual Exploitation     103
Name: count, dtype: int64


In [36]:
combined_df.to_csv(
    "/content/toxic_data_balanced_500.csv",
    index=False
)

In [49]:
import pandas as pd

# Create one unified label column
combined_df["final_label"] = (
    combined_df["final_label"]
    .fillna(combined_df["Toxic Category"])
)

# Check remaining nulls
print("Remaining null labels:",
      combined_df["final_label"].isna().sum())

# Check distribution
print("\nFinal label distribution:")
print(combined_df["final_label"].value_counts())

Remaining null labels: 0

Final label distribution:
final_label
Safe                         1646
Violent Crimes               1610
Suicide & Self-Harm          1211
Child Sexual Exploitation     588
Sex-Related Crimes            578
Elections                     568
Non-Violent Crimes            553
Unknown S-Type                504
unsafe                        500
Name: count, dtype: int64


In [51]:
combined_df["final_label"].isnull().sum()

np.int64(0)

In [52]:
combined_df.columns

Index(['text', 'Toxic Category', 'source', 'final_label'], dtype='object')

In [53]:
combined_df.drop(["Toxic Category", "source"], axis=1, inplace=True)

In [60]:
combined_df.head()

,Toxic Category,final_label
0,A child playing in a sunny meadow.,Safe
1,A family enjoying a picnic in the park.,Safe
2,A child playing in a sunny meadow.,Safe
3,Police tape across a crime scene at night.,Violent Crimes
4,A child playing in a sunny meadow.,Safe


In [61]:
combined_df.rename(columns={"Toxic Category": "text", "final_label":"Toxic Category"}, inplace=True)
combined_df.head()

,text,Toxic Category
0,A child playing in a sunny meadow.,Safe
1,A family enjoying a picnic in the park.,Safe
2,A child playing in a sunny meadow.,Safe
3,Police tape across a crime scene at night.,Violent Crimes
4,A child playing in a sunny meadow.,Safe


In [62]:
combined_df.to_csv(
    "/content/toxic_data_final.csv",
    index=False
)